[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PTB-MR/qMRData/blob/main/qualitative_imaging.ipynb)

In [ ]:
import importlib

if not importlib.util.find_spec("mrpro"):
    %pip install mrpro[notebooks]

## Data loader

To make it easier to load the data from the different dataset we define a data loader here.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Callable
import numpy as np

SCANNER_SUBJECTS = {
    "siemens_freemax_knee_2026": {"Left Knee", "Right Knee"},
    "physio_mr_knee_2026": {
        "Subject2-LeftKnee",
        "Subject2-RightKnee",
        "Subject5-LeftKnee",
        "Subject5-RightKnee",
    },
}


@dataclass
class FileConfig:
    raw_filename: Path
    image_folder: Path


@dataclass
class DatasetConfig:
    subject: str
    scanner: str

    t1w: FileConfig
    t2w: FileConfig
    pdw: FileConfig

    raw_adapt_orientation: Callable[[np.ndarray], np.ndarray] = lambda x: x
    image_adapt_orientation: Callable[[np.ndarray], np.ndarray] = lambda x: x


def get_dataset_config(scanner: str, subject: str) -> DatasetConfig:
    if scanner not in SCANNER_SUBJECTS:
        raise ValueError(f"Unknown scanner: {scanner}")

    if subject not in SCANNER_SUBJECTS[scanner]:
        raise ValueError(
            f"Subject {subject} not valid for scanner {scanner}. "
            f"Valid options: {sorted(SCANNER_SUBJECTS[scanner])}"
        )

    if scanner == "siemens_freemax_knee_2026":
        base_dir = Path(
            "/Users/kolbit01/Documents/Data/Claudia_055/Test 03 DB Knee Protocol/"
        )
        t1w = FileConfig(
            raw_filename=list((base_dir / "raw" / subject).glob("*t1_tse_sag*.mrd"))[0],
            image_folder=list(
                (base_dir / "dicom" / subject / "T1w").glob("*t1_tse_sag*")
            )[0],
        )
        t2w = FileConfig(
            raw_filename=list((base_dir / "raw" / subject).glob("*t2_tse_sag*.mrd"))[0],
            image_folder=list(
                (base_dir / "dicom" / subject / "T2w").glob("*t2_tse_sag*")
            )[0],
        )
        pdw = FileConfig(
            raw_filename=list((base_dir / "raw" / subject).glob("*pd_tse_sag*.mrd"))[0],
            image_folder=list(
                (base_dir / "dicom" / subject / "PD").glob("*pd_tse_sag*")
            )[0],
        )

        def raw_adapt_orientation(x):
            return np.rot90(x, 1)[:, ::-1]

        def image_adapt_orientation(x):
            return np.rot90(x, -1)

    elif scanner == "physio_mr_knee_2026":
        base_dir = Path(
            "/Users/kolbit01/Documents/Data/Joseba/Qualitative knee images i3M/"
        )
        t1w = FileConfig(
            raw_filename=list((base_dir / subject).glob("*-T1*.h5"))[0],
            image_folder=base_dir / subject / "Dicom" / "T1w",
        )
        t2w = FileConfig(
            raw_filename=list((base_dir / subject).glob("*-T2*.h5"))[0],
            image_folder=base_dir / subject / "Dicom" / "T2w",
        )
        pdw = FileConfig(
            raw_filename=list((base_dir / subject).glob("*-PD*.h5"))[0],
            image_folder=base_dir / subject / "Dicom" / "PD",
        )

        def raw_adapt_orientation(x):
            return np.rot90(x, -1)

        def image_adapt_orientation(x):
            return np.rot90(x, -1)

    return DatasetConfig(
        subject=subject,
        scanner=scanner,
        t1w=t1w,
        t2w=t2w,
        pdw=pdw,
        raw_adapt_orientation=raw_adapt_orientation,
        image_adapt_orientation=image_adapt_orientation,
    )

## Qualitative Image Reconstruction

Qualitative image reconstruction does not require any sequence specific signal models and so we can use the same image reconstruction code to obtain image data from multiple scanners and field strengths. 
Here we will focus on T1-weighted, T2-weighted and proton density (PD)-weighted images. All the parameters required for image reconstruction are part of the (ISMR)MRD raw data file. 

In [ ]:
import numpy as np
import torch
from mrpro.algorithms.reconstruction import DirectReconstruction
from mrpro.data import KData, IData, CsmData
from mrpro.data.traj_calculators import KTrajectoryCartesian

## Select the data files

In [ ]:
dataset = get_dataset_config("siemens_freemax_knee_2026", "Left Knee")
dataset = get_dataset_config("physio_mr_knee_2026", "Subject5-LeftKnee")

## Reconstruct images

In [ ]:
kdata = KData.from_file(dataset.t1w.raw_filename, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
t1w = recon(kdata)

kdata = KData.from_file(dataset.t2w.raw_filename, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
t2w = recon(kdata)

kdata = KData.from_file(dataset.pdw.raw_filename, trajectory=KTrajectoryCartesian())
csm = CsmData.from_kdata_inati(kdata, downsampled_size=64)
recon = DirectReconstruction(kdata, csm=csm)
pdw = recon(kdata)

## Visualise results

In [ ]:
import matplotlib.pyplot as plt
from typing import Callable


def show_image(t1w: IData, t2w: IData, pdw: IData, adapt_orientation: Callable) -> None:
    """Show the qualitative images."""
    _, ax = plt.subplots(3, 3, figsize=(8, 8))

    for cax in ax.flatten():
        cax.set_axis_off()

    def plot_multi_slice_image(ax, idata, title):
        """Plot three slices of M2D/3D image."""
        # Ensure the slices are in the correct order
        if idata.shape[0] > 1:
            sort_idx = torch.argsort(idata.header.position.x.squeeze())
            idata = idata[sort_idx]
        idim = idata.data.squeeze().shape[-3]
        vmax = 0.8 * idata.data.abs().max()
        for cax, slice_idx in zip(
            ax, [int(idim // 2 - idim // 6), idim // 2, int(idim // 2 + idim // 6)]
        ):
            cax.imshow(
                adapt_orientation(
                    idata.data.squeeze()[slice_idx].abs().numpy(force=True)
                ),
                cmap="gray",
                vmin=0,
                vmax=vmax,
            )
        ax[1].set_title(title)

    plot_multi_slice_image(ax[0], t1w, "T1-weighted")
    plot_multi_slice_image(ax[1], t2w, "T2-weighted")
    plot_multi_slice_image(ax[2], pdw, "PD-weighted")

    plt.tight_layout()
    plt.show()

In [ ]:
show_image(t1w, t2w, pdw, dataset.raw_adapt_orientation)

## Load Dicom data

In [ ]:
t1w_dicom = IData.from_dicom_folder(dataset.t1w.image_folder)
t2w_dicom = IData.from_dicom_folder(dataset.t2w.image_folder)
pdw_dicom = IData.from_dicom_folder(dataset.pdw.image_folder)

## Visualise Dicom images

In [ ]:
show_image(t1w_dicom, t2w_dicom, pdw_dicom, dataset.image_adapt_orientation)